# Query Yahoo Iceberg Tables

This notebook shows how to query the Yahoo Fantasy Iceberg tables written by this repo.

It assumes the default Iceberg catalog settings from `nfl.yahoo_fantasy.storage.iceberg.IcebergCatalogConfig`:
- Catalog: SQL (`sqlite:///iceberg_catalog.db`)
- Warehouse: `./warehouse`
- Namespaces: `yahoo_common` and `yhnfl`

In [7]:
from pathlib import Path
import sys

import polars as pl
from pyiceberg.catalog import load_catalog

In [8]:
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [ ]:
cwd = Path.cwd()

# Resolve repository root from cwd by walking up to pyproject.toml.
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            if current.name.lower() == "examples":
                raise RuntimeError("Invalid project root resolved to 'examples'.")
            return current
        current = current.parent
    raise RuntimeError("Cannot locate project root. Expected pyproject.toml in repository root.")

project_root = find_project_root(cwd)

CATALOG_URI = f"sqlite:///{project_root / 'iceberg_catalog.db'}"
WAREHOUSE = str(project_root / "warehouse")

catalog = load_catalog(
    "yahoo",
    type="sql",
    uri=CATALOG_URI,
    warehouse=WAREHOUSE,
)

print("Catalog loaded.")
print("Catalog URI:", CATALOG_URI)
print("Warehouse:", Path(WAREHOUSE).resolve())

Catalog loaded.
Catalog URI: sqlite:///c:\Users\EricTruett\nfl\iceberg_catalog.db
Warehouse: C:\Users\EricTruett\nfl\warehouse


In [11]:
def list_tables(namespace: str) -> list[str]:
    try:
        return [f"{ns}.{tbl}" for ns, tbl in catalog.list_tables(namespace)]
    except Exception as exc:
        print(f"Could not list tables for namespace '{namespace}': {exc}")
        return []

for ns in ["yahoo_common", "yhnfl"]:
    tables = list_tables(ns)
    print(f"\n{ns}:")
    if tables:
        for t in tables:
            print(" -", t)
    else:
        print(" (no tables found)")


yahoo_common:
 - yahoo_common.draft_pick
 - yahoo_common.league
 - yahoo_common.player
 - yahoo_common.team
 - yahoo_common.transaction

yhnfl:
 - yhnfl.matchups
 - yhnfl.player_stats_weekly
 - yhnfl.roster_entries
 - yhnfl.standings


In [12]:
def load_table_as_polars(table_identifier: str) -> pl.DataFrame:
    """Load an Iceberg table into a Polars DataFrame."""
    table = catalog.load_table(table_identifier)
    arrow_table = table.scan().to_arrow()
    return pl.from_arrow(arrow_table)

def maybe_load(table_identifier: str) -> pl.DataFrame | None:
    try:
        return load_table_as_polars(table_identifier)
    except Exception as exc:
        print(f"Skipping '{table_identifier}': {exc}")
        return None

## Example 1: League and Team Snapshot

This joins `yahoo_common.league` and `yahoo_common.team` to show team context per league.

In [13]:
league_df = maybe_load("yahoo_common.league")
team_df = maybe_load("yahoo_common.team")

if league_df is not None and team_df is not None:
    league_team = (
        team_df.join(
            league_df.select(["league_key", "season", "league_name"]),
            on="league_key",
            how="left",
        )
        .select(["league_key", "season", "league_name", "team_key", "team_name", "owner_name"])
        .sort(["season", "league_key", "team_key"])
    )
    league_team.head(20)
else:
    print("Required tables are not available yet. Run the Yahoo pipeline first.")

c:\Users\EricTruett\miniconda3\envs\dbxconnect\Lib\site-packages\pyiceberg\io\__init__.py:362: UserWarning: No preferred file implementation for scheme: c
  if file_io := _infer_file_io_from_scheme(warehouse_location, properties):


Skipping 'yahoo_common.league': [WinError 3] Failed to open local file 'c:/Users/EricTruett/nfl/examples/warehouse/yahoo_common/league/metadata/00001-7ec1f76b-1f82-45e2-b5f4-676b4667b359.metadata.json'. Detail: [Windows error 3] The system cannot find the path specified.

Skipping 'yahoo_common.team': [WinError 3] Failed to open local file 'c:/Users/EricTruett/nfl/examples/warehouse/yahoo_common/team/metadata/00001-8fb6bfe8-c232-40a9-9326-25cdf4b0c86e.metadata.json'. Detail: [Windows error 3] The system cannot find the path specified.

Required tables are not available yet. Run the Yahoo pipeline first.


c:\Users\EricTruett\miniconda3\envs\dbxconnect\Lib\site-packages\pyiceberg\io\__init__.py:362: UserWarning: No preferred file implementation for scheme: c
  if file_io := _infer_file_io_from_scheme(warehouse_location, properties):


## Example 2: NFL Standings Summary

This reads `yhnfl.standings` and ranks teams by wins and points for.

In [14]:
standings_df = maybe_load("yhnfl.standings")

if standings_df is not None:
    standings_summary = (
        standings_df
        .sort(["season", "wins", "points_for"], descending=[False, True, True])
        .select([
            "league_key",
            "season",
            "team_key",
            "rank",
            "wins",
            "losses",
            "ties",
            "points_for",
            "points_against",
        ])
    )
    standings_summary.head(20)
else:
    print("Table yhnfl.standings not found.")

Skipping 'yhnfl.standings': [WinError 3] Failed to open local file 'c:/Users/EricTruett/nfl/examples/warehouse/yhnfl/standings/metadata/00001-2d7e8ed5-1486-4a1f-8f72-68df4332e5e8.metadata.json'. Detail: [Windows error 3] The system cannot find the path specified.

Table yhnfl.standings not found.


## Example 3: Weekly Fantasy Points by Team

This combines `yhnfl.player_stats_weekly` with `yhnfl.roster_entries` to summarize total fantasy points by week and team.

In [15]:
stats_df = maybe_load("yhnfl.player_stats_weekly")
roster_df = maybe_load("yhnfl.roster_entries")

if stats_df is not None and roster_df is not None:
    weekly_points = (
        stats_df.join(
            roster_df.select(["league_key", "week", "team_key", "player_key"]),
            on=["league_key", "week", "player_key"],
            how="left",
        )
        .group_by(["league_key", "season", "week", "team_key"])
        .agg([
            pl.sum("points").alias("team_points"),
            pl.n_unique("player_key").alias("players_counted"),
        ])
        .sort(["season", "week", "team_points"], descending=[False, False, True])
    )
    weekly_points.head(30)
else:
    print("Required NFL weekly tables not found.")

Skipping 'yhnfl.player_stats_weekly': [WinError 3] Failed to open local file 'c:/Users/EricTruett/nfl/examples/warehouse/yhnfl/player_stats_weekly/metadata/00001-752e3fa1-1ab4-4e7d-8a75-f4463c8c526c.metadata.json'. Detail: [Windows error 3] The system cannot find the path specified.

Skipping 'yhnfl.roster_entries': [WinError 3] Failed to open local file 'c:/Users/EricTruett/nfl/examples/warehouse/yhnfl/roster_entries/metadata/00001-9c406628-b001-44de-8b2a-f82f7a4cbc4d.metadata.json'. Detail: [Windows error 3] The system cannot find the path specified.

Required NFL weekly tables not found.


## Optional: Quick SQL on Polars DataFrames

If you prefer SQL syntax, register loaded frames in a Polars SQL context.

In [17]:
ctx = pl.SQLContext()

if league_df is not None:
    ctx.register("league", league_df)
if team_df is not None:
    ctx.register("team", team_df)

if league_df is not None and team_df is not None:
    sql_result = ctx.execute("""
        SELECT
            l.season,
            l.league_name,
            COUNT(*) AS team_count
        FROM team t
        JOIN league l ON t.league_key = l.league_key
        GROUP BY l.season, l.league_name
        ORDER BY l.season DESC, l.league_name
    """)
    sql_result
else:
    print("Load league/team tables first before running this cell.")

Load league/team tables first before running this cell.
